In [50]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
)

# Load data
data = np.loadtxt(r"C:\Users\Sam\Desktop\ML\data\Data_err.npt")
y_real = data[:, 0]
y_pred = data[:, 1]

# Split into train/test
split_idx = int(len(y_real) * 0.8)
y_real_train, y_real_test = y_real[:split_idx], y_real[split_idx:]
y_pred_train, y_pred_test = y_pred[:split_idx], y_pred[split_idx:]

# Metric function
def get_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    gmean = np.sqrt(recall_score(y_true, y_pred, zero_division=0) * specificity)
    return {
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F-measure": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "G-mean": gmean,
        "Specificity": specificity,
    }

# Compute metrics
metrics_all = get_metrics(y_real, y_pred)
metrics_train = get_metrics(y_real_train, y_pred_train)
metrics_test = get_metrics(y_real_test, y_pred_test)

# Create main metrics DataFrame
df_main = pd.DataFrame(
    [
        ["All", *metrics_all.values()],
        ["Train", *metrics_train.values()],
        ["Test", *metrics_test.values()],
    ],
    columns=["Set", "Recall", "Accuracy", "F-measure", "Precision", "G-mean", "Specificity"],
)

# Compute per-class metrics using average=None
precision_per_class = precision_score(y_real, y_pred, average=None, zero_division=0)
recall_per_class = recall_score(y_real, y_pred, average=None, zero_division=0)
f1_per_class = f1_score(y_real, y_pred, average=None, zero_division=0)
f2_per_class = fbeta_score(y_real, y_pred, average=None, beta=2, zero_division=0)

# Accuracy per class
accuracy_per_class = []
specificity_per_class = []
gmean_per_class = []
for cls in np.unique(y_real):
    idx = y_real == cls
    acc = accuracy_score(y_real[idx], y_pred[idx])
    cm = confusion_matrix(y_real[idx], y_pred[idx], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    rec = recall_score(y_real[idx], y_pred[idx], zero_division=0)
    gmean = np.sqrt(rec * spec)
    accuracy_per_class.append(acc)
    specificity_per_class.append(spec)
    gmean_per_class.append(gmean)

# Build DataFrame
df_class = pd.DataFrame(
    {
        "Class": np.unique(y_real).astype(int),
        "Recall": recall_per_class,
        "Accuracy": accuracy_per_class,
        "F-measure": f1_per_class,
        "Precision": precision_per_class,
        "G-mean": gmean_per_class,
        "Specificity": specificity_per_class,
    }
)

# Display both tables
print("Main Metrics Table:")
print(df_main.to_string(index=False))
print("\nPer-Class Metrics Table:")
print(df_class.to_string(index=False))
# ROC and AUC
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_real, y_pred)
roc_auc = auc(fpr, tpr)
auc_column = [""] * (len(fpr) - 1) + [round(roc_auc, 3)]
roc_df = pd.DataFrame({"FPR": fpr, "TPR": tpr, "AUC": auc_column})
# print("\nROC Curve Data:")
# print(roc_df)

# Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_real, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
# print("\nConfusion Matrix:")
# print(cm_df)

# Plot heatmap
# sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
# plt.title("Confusion Matrix Heatmap")
# plt.show()

Main Metrics Table:
  Set   Recall  Accuracy  F-measure  Precision   G-mean  Specificity
  All 0.993017  0.994132   0.992093   0.991171 0.993902     0.994789
Train 0.993015  0.994175   0.992149   0.991284 0.993936     0.994858
 Test 0.993023  0.993960   0.991870   0.990719 0.993768     0.994513

Per-Class Metrics Table:
 Class   Recall  Accuracy  F-measure  Precision  G-mean  Specificity
     0 0.994789  0.994789   0.995335   0.995881     0.0     0.994789
     1 0.993017  0.993017   0.992093   0.991171     0.0     0.000000


In [46]:
df_main.to_clipboard(index=False)


In [47]:
df_class.to_clipboard(index=False)


In [48]:
roc_df.to_clipboard(index=False)


In [49]:
cm_df.to_clipboard(index=False)
